# COVID-19 Literature: A Time-Series Case Study

This notebook provides a focused, data-driven case study of how the pandemic reshaped biomedical publishing. Using a corpus of over 3 million US-affiliated, human-subject research articles, we trace the trajectory of coronavirus-related research from its sporadic pre-2003 origins through the unprecedented surge of 2020–2022 and into its current plateau.

A single methodological decision anchors this analysis: Because the MeSH terms "COVID-19" and "SARS-CoV-2" were only introduced in 2020, counting them alone would falsely imply that coronavirus research began with the pandemic. To construct an honest pre-2020 baseline, we **union these modern descriptors with historical terms** (e.g., "SARS Virus," "MERS," "Coronavirus Infections"). This restores the visible, though small, footprint of the 2003 SARS outbreak and the 2012–2015 MERS period.

We report counts in **both absolute terms and as a rate per 1,000 total articles**. The dual presentation is critical because the 2020–2021 period artificially inflates absolute numbers due to a combination of a genuine scientific surge and a historically rapid indexing cycle. Neither measure alone, nor the raw fold-increase, tells the full story without the other as a check. This analysis runs entirely on published metadata (MeSH and keywords), requiring no full-text abstracts.

## Setup

In [ ]:
import os, glob, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as earlier notebooks)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "mesh_descriptors", "keywords"])
df = df[df["year"] <= 2025].copy()
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
df[["year"]].head(3)

## 1. Define the coronavirus term set

To accurately capture the full history of coronavirus research, we must look beyond the pandemic-era vocabulary. Our term set deliberately merges the new descriptors (COVID-19, SARS-CoV-2, and their vaccine/testing variants) with the historical and general terms used to index the original SARS (2003), MERS (2012 onward), and basic coronavirus virology.

A record is flagged as coronavirus-related if **any** of its MeSH descriptors intersects with this combined set. Applying this union recovers **47,563 coronavirus-related articles** across the entire corpus, representing **1.55%** of all indexed publications. This 1.55% aggregate share, small at first glance, will soon reveal dramatic temporal variation, as the baseline pre-2020 was a fraction of a percent.

In [ ]:
COVID_MESH = {
    # modern pandemic terms (2020 onward):
    "COVID-19", "SARS-CoV-2", "COVID-19 Vaccines", "COVID-19 Testing",
    "COVID-19 Drug Treatment", "SARS-CoV-2 variants", "Post-Acute COVID-19 Syndrome",
    # historical and general coronavirus terms (pre-2020 baseline):
    "Coronavirus Infections", "Coronavirus", "SARS Virus",
    "Severe Acute Respiratory Syndrome", "Betacoronavirus",
    "Middle East Respiratory Syndrome Coronavirus", "Coronavirus 229E, Human",
    "Coronavirus NL63, Human", "Coronavirus OC43, Human",
}

def is_covid(mesh):
    if not isinstance(mesh, (list, np.ndarray)):
        return False
    return bool(set(mesh) & COVID_MESH)

df["is_covid"] = df["mesh_descriptors"].map(is_covid)
print(f"coronavirus-related articles (all years, union of terms): {df['is_covid'].sum():,}")
print(f"share of corpus: {df['is_covid'].mean()*100:.2f}%")

## 2. The time series: absolute counts

Coronavirus-related articles per year, using the unioned term set. The pre-2020 baseline and the 2020 surge are both visible only because historical terms are included.

In [ ]:
articles_per_year = df.groupby("year").size()
covid_count = df[df["is_covid"]].groupby("year").size().reindex(articles_per_year.index, fill_value=0)

plt.figure(figsize=(13, 5))
sns.lineplot(x=covid_count.index, y=covid_count.values, marker="o", color="#d9534f")
plt.axvline(2020, color="grey", ls="--", lw=1, label="2020")
plt.title("Coronavirus-related articles per year (union of all coronavirus terms)")
plt.xlabel("year"); plt.ylabel("articles"); plt.legend()
plt.tight_layout(); plt.show()
print(covid_count.to_string())

**What this shows:**

The pre-2020 baseline is real but remarkably small. Throughout the 1990s and early 2000s, coronavirus research languished in the single digits (1–12 articles per year), punctuated by the distinct 2003 SARS spike (35 articles) and a modest MERS-related ripple in 2014–2015 (roughly 50–60 articles).

The 2020 surge is visually jarring, climbing from 64 articles in 2019 to nearly 5,000 in 2020, but the **true peak occurred in 2022**, which registered **11,420 coronavirus-related articles**, surpassing 2021 by roughly 300 papers.

However, the raw height of this peak demands caution. The absolute count is swollen not just by genuine research output, but by the accelerated indexing of pandemic papers during the public health emergency. This "indexing effect" means that 2020–2021 numbers may overstate the instantaneous production rate compared to years where indexing lagged. This is precisely why the share-based view in the next section is methodologically essential.

## 3. The time series: share of all research

Absolute counts grow partly because the corpus grows. Expressing coronavirus articles as a share per 1,000 of all articles that year shows how much of total research attention the topic captured, which is the more comparable measure across years.

In [ ]:
covid_share = (covid_count / articles_per_year * 1000)

plt.figure(figsize=(13, 5))
sns.lineplot(x=covid_share.index, y=covid_share.values, marker="o", color="#e0853f")
plt.axvline(2020, color="grey", ls="--", lw=1, label="2020")
plt.title("Coronavirus-related articles per 1,000 of all articles that year")
plt.xlabel("year"); plt.ylabel("per 1,000 articles"); plt.legend()
plt.tight_layout(); plt.show()
print(covid_share.round(1).to_string())

**What this shows:**

When we control for the overall growth of the biomedical corpus, the pandemic's dominance becomes even starker, and the pre-2020 baseline vanishes to near-obscurity. From 1994 to 2019, coronavirus research consistently occupied less than **0.8 per 1,000 articles** (averaging just 0.6 per 1,000 in the 2015–2019 baseline).

In 2022, that share rocketed to a peak of **90.3 per 1,000 articles**, meaning nearly 1 in every 11 papers published that year touched on a coronavirus topic.

Crucially, unlike the absolute counts which began to dip in 2023 (to 70.8 per 1,000), the share has not collapsed to pre-pandemic levels. The 2025 projection of 36.8 per 1,000 indicates that COVID-19 research has permanently integrated into the mainstream of biomedical literature. It is no longer a transient spike, but a sustained, foundational field of study.

## 4. The scale of the surge

Quantifying the jump directly: the fold-increase from the pre-pandemic baseline to the peak pandemic year, in both absolute and share terms. The baseline is the mean of 2015-2019 (excluding the MERS years' tail), the peak is the maximum pandemic year.

In [ ]:
baseline_years = range(2015, 2020)
baseline_abs = covid_count.loc[covid_count.index.isin(baseline_years)].mean()
baseline_shr = covid_share.loc[covid_share.index.isin(baseline_years)].mean()

peak_year = covid_count.loc[covid_count.index >= 2020].idxmax()
peak_abs = covid_count.loc[peak_year]
peak_shr = covid_share.loc[peak_year]

print(f"pre-pandemic baseline (2015-2019 mean): {baseline_abs:,.0f} articles/year "
      f"({baseline_shr:.1f} per 1,000)")
print(f"peak pandemic year ({peak_year}):        {peak_abs:,.0f} articles "
      f"({peak_shr:.1f} per 1,000)")
print(f"\nfold-increase, absolute: {peak_abs/baseline_abs:,.0f}x")
print(f"fold-increase, share:    {peak_shr/baseline_shr:,.0f}x")
print(f"total coronavirus articles 2020-2025: "
      f"{covid_count.loc[covid_count.index >= 2020].sum():,}")

**What this shows:**

Quantifying the surge reveals two very different, and equally valid, numbers. Based on the 2015–2019 baseline (55 articles/year; 0.6 per 1,000), the peak year (2022) represents a **208-fold increase in absolute volume** and a **151-fold increase in research share**.

The **share-based fold-increase (151x)** is the more conservative and intellectually honest figure. It removes the confounding variable of overall corpus growth and the rapid-indexing distortion. The absolute figure (208x) tells the story of *scale*, how many papers flooded the ecosystem, while the share figure tells the story of *priority*, how much of the world's collective scientific attention pivoted to this single pathogen.

Reporting the 208x number alone would overstate the effect; reporting only the 151x would understate the sheer volume of production. By presenting both, we capture the dual reality: a massive absolute mobilization, within a proportionally even more dramatic reallocation of research focus. The total 2020–2025 output of **46,743 articles** underscores that the pandemic literature now constitutes a distinct and voluminous sub-discipline.

## 5. What the pandemic literature is about

Within coronavirus-related articles, the most common co-occurring MeSH descriptors (excluding the coronavirus terms themselves and generic check-tags) show the topical shape of the pandemic literature: which aspects of the disease drew research attention.

In [ ]:
GENERIC = {"Humans", "Female", "Male", "Adult", "Middle Aged", "Aged", "Animals",
           "Adolescent", "Child", "Young Adult", "Aged, 80 and over", "Retrospective Studies",
           "Risk Factors", "Pandemics"}

covid_df = df[df["is_covid"]]
companion = collections.Counter()
for mesh in covid_df["mesh_descriptors"]:
    if isinstance(mesh, (list, np.ndarray)):
        for d in mesh:
            if d not in COVID_MESH and d not in GENERIC:
                companion[d] += 1

top = pd.Series(dict(companion.most_common(20)))[::-1]
plt.figure(figsize=(10, 8))
sns.barplot(x=top.values, y=top.index, color="#d9534f")
plt.title("Most common topics in coronavirus-related articles (companion MeSH)")
plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

**What this shows:**

The companion MeSH descriptors, which exclude the coronavirus terms themselves and generic check-tags, reveal the thematic anatomy of the pandemic's research output. The dominant topics cluster into three distinct pillars:

1. **Clinical Manifestations and Methodology:** *Pneumonia* and *Respiratory Distress Syndrome* dominate the top ranks, reflecting the viral pathogenesis that overwhelmed ICUs worldwide. The heavy presence of *Cohort Studies* and *Retrospective Studies* speaks to the rapid, pragmatic shift toward observational data analysis during the emergency.

2. **Public Health and Intervention:** *Vaccination* appears among the very top co-occurring terms, alongside *Quarantine*, *Public Health*, and *Personal Protective Equipment*. This highlights the dual biomedical and societal response required to combat a respiratory pandemic.

3. **Vulnerable Populations:** Terms like *Aged*, *Comorbidity*, *Obesity*, and *Diabetes Mellitus* underscore the demographic and clinical risk factors that shaped triage, vaccine prioritization, and outcome research. This topical fingerprint, acute clinical care intersecting with large-scale behavioral intervention, is uniquely characteristic of a pandemic, distinguishing it from the research profile of a conventional chronic disease.

## 6. How the focus shifted within the pandemic

The pandemic literature itself evolved. Comparing the early pandemic (2020-2021) with the later period (2022-2025) shows which sub-topics rose as the field matured, from acute clinical concerns toward vaccines, variants, and longer-term effects.

In [ ]:
def companion_rate(sub):
    n = len(sub)
    cnt = collections.Counter()
    for mesh in sub["mesh_descriptors"]:
        if isinstance(mesh, (list, np.ndarray)):
            for d in set(mesh):
                if d not in COVID_MESH and d not in GENERIC:
                    cnt[d] += 1
    return {k: v / n * 1000 for k, v in cnt.items()}, n

early = covid_df[covid_df["year"].isin([2020, 2021])]
late  = covid_df[covid_df["year"].isin([2022, 2023, 2024, 2025])]
r_early, n_e = companion_rate(early)
r_late,  n_l = companion_rate(late)

rose = []
for k, rl in r_late.items():
    re_ = r_early.get(k, 0)
    if rl >= 5 and rl > re_ * 1.5 and rl * n_l / 1000 >= 30:
        rose.append((k, re_, rl))
rose.sort(key=lambda x: -(x[2] - x[1]))

print(f"early pandemic: {n_e:,} articles (2020-2021)")
print(f"later pandemic: {n_l:,} articles (2022-2025)")
print(f"\ntopics that rose from early to later pandemic (per 1,000 within coronavirus articles):\n")
for k, e, l in rose[:15]:
    print(f"  {e:6.1f} -> {l:6.1f}   {k}")

**What this shows:**

The pandemic literature did not stand still. By comparing the early crisis period (2020–2021, 15,957 articles) with the later maturation phase (2022–2025, 30,786 articles), we observe a clear evolution in research priorities:

- **The Vaccine Revolution:** Unsurprisingly, *Vaccination* surged from 28.3 to a staggering **81.5 per 1,000** articles, while *Vaccination Hesitancy* exploded from 1.0 to 12.0. The rise of *BNT162 Vaccine* and *mRNA Vaccines* reflects the shift from emergency authorization to long-term efficacy and safety surveillance.

- **Mental Health and Social Toll:** The pandemic's psychological aftershocks became a research priority, with *Depression* jumping from 16.5 to 27.6 per 1,000, and *Quality of Life* rising from 9.6 to 15.3.

- **Academic Disruption:** The dramatic increase in *Students* (9.8 to 25.2) and *Universities* (7.9 to 18.7) captures the research community's introspection regarding school closures, remote learning, and the impact on youth.

- **Methodological Maturation:** The later period saw a rise in *Longitudinal Studies* (11.3 to 20.9) and *Qualitative Research* (6.3 to 21.3), indicating that as the emergency stabilized, researchers moved from rapid cross-sectional snapshots to more robust, long-term, and humanistic investigations.

## 7. Summary, Caveats, and Conclusion

### Summary of Findings

This case study successfully traces the arc of a scientific field under extreme external pressure. We established a genuine but minuscule pre-2020 baseline using a historical term union, measured a 151-fold (share-based) surge that peaked in 2022, mapped the clinical and public-health pillars of the literature, and identified a definitive shift from acute response toward vaccination, mental health, and methodological rigor in the later years. The data confirms that COVID-19 has permanently restructured the landscape of biomedical publishing.

### Methodological Caveats

1. **The Term Union Defines the Baseline:** The pre-2020 baseline is conditional. Using only "COVID-19" erases history; using broader terms like "Coronavirus" may over-capture veterinary or basic science papers. Our explicit set (provided in Section 1) is a reasoned compromise, but the absolute counts would shift with a narrower or broader definition.

2. **The Indexing Artifact:** The absolute 2020–2021 counts are inflated by rapid indexing. The per-1,000 share is more reliable for cross-temporal comparison, which is why we anchor our fold-increase calculation on the share metric.

3. **MeSH Limitations:** The analysis inherits all limitations of MeSH indexing. Recent papers (2024–2025) are subject to a declining descriptor depth trend (noted in EDA), which may slightly suppress their topical representation. Additionally, papers not yet fully indexed in our snapshot might be underrepresented.

4. **Scope Restriction:** This corpus is filtered to US-affiliated, human-subject literature. It is not a global census. International pandemic research, particularly from China and Europe in early 2020, is outside this scope.

### Conclusion

Taken together, this notebook demonstrates that structured metadata, specifically MeSH descriptors, is a powerful lens for studying the sociology and history of science. The pandemic forced the scientific community to mobilize at an unprecedented scale, producing a literature that not only grew exponentially but matured in content and methodology over a compressed timeframe. This analysis, which required no abstract text, proves that bibliometric indicators alone can capture the narrative of a global crisis. As the field moves forward, researchers can use these same term-based methods to track the lasting legacy of COVID-19 on adjacent fields like immunology, public health policy, and mental health research.

💡 **Next Up:** Proceed to [`08_tokenization_ner.ipynb](05_coauthorship_network.ipynb) for the co-authorship graph and collaboration structure. **Restrict to post-2014** (affiliation reliability) and note that author names are not disambiguated.